In [1]:
import os
import torch
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader, Subset, random_split
from torchvision.io import read_image
from sklearn.metrics import accuracy_score, recall_score
import torch.nn as nn
import torch.nn.functional as F
from tqdm import tqdm
import itertools
import random

# === Device ===
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


Using device: cpu


In [2]:
# === Custom Dataset ===
class CustomImageDataset(Dataset):
    def __init__(self, root_dir, transform=None, limit=10000):
        self.samples = []
        self.transform = transform
        for patient in os.listdir(root_dir):
            patient_dir = os.path.join(root_dir, patient)
            if not os.path.isdir(patient_dir):
                continue
            for label in ['0', '1']:
                label_dir = os.path.join(patient_dir, label)
                if os.path.isdir(label_dir):
                    for fname in os.listdir(label_dir):
                        img_path = os.path.join(label_dir, fname)
                        self.samples.append((img_path, int(label)))

        random.shuffle(self.samples)
        self.samples = self.samples[:limit]

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        image = read_image(path).float() / 255.0
        if self.transform:
            image = self.transform(image)
        return image, label


In [3]:
# === Transforms ===
transform = transforms.Compose([
    transforms.Resize((64, 64)),
])

In [4]:
# === Dataset & Split ===
root_dir = "../data/image50"  # Remplacer par le bon chemin
full_dataset = CustomImageDataset(root_dir, transform=transform)
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])

In [5]:
# === CNN Model ===
class SimpleCNN(nn.Module):
    def __init__(self, conv_layers, fc_units, dropout):
        super(SimpleCNN, self).__init__()
        layers = []
        in_channels = 3
        for _ in range(conv_layers):
            layers.append(nn.Conv2d(in_channels, 16, kernel_size=3, padding=1))
            layers.append(nn.ReLU())
            layers.append(nn.MaxPool2d(2))
            in_channels = 16
        self.conv = nn.Sequential(*layers)

        self.feature_size = 16 * (64 // (2 ** conv_layers)) ** 2
        self.fc1 = nn.Linear(self.feature_size, fc_units)
        self.dropout = nn.Dropout(dropout)
        self.fc2 = nn.Linear(fc_units, 2)

    def forward(self, x):
        x = self.conv(x)
        x = x.view(x.size(0), -1)
        x = self.dropout(F.relu(self.fc1(x)))
        return self.fc2(x)


In [6]:
# === Param Grid ===
param_grid = {
    "conv_layers": [1, 2, 3],
    "fc_units": [128, 256],
    "lr": [0.01, 0.001],
    "dropout": [0.0, 0.3],
    "batch_size": [32]
}

best_score = 0
best_model = None

# === Grid Search ===
for conv_layers, fc_units, lr, dropout, batch_size in itertools.product(
    param_grid["conv_layers"],
    param_grid["fc_units"],
    param_grid["lr"],
    param_grid["dropout"],
    param_grid["batch_size"]):

    print(f"\n🔍 Testing conv_layers={conv_layers}, fc_units={fc_units}, lr={lr}, dropout={dropout}, batch_size={batch_size}")

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size)

    model = SimpleCNN(conv_layers, fc_units, dropout).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    # === Training ===
    model.train()
    loop = tqdm(train_loader, desc="Training", leave=False)
    for images, labels in loop:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        loop.set_postfix(loss=loss.item())

    # === Evaluation ===
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            preds = outputs.argmax(dim=1).cpu()
            all_preds.extend(preds.numpy())
            all_labels.extend(labels.cpu().numpy())

    acc = accuracy_score(all_labels, all_preds)
    recall = recall_score(all_labels, all_preds)
    avg_score = (acc + recall) / 2

    print(f"✅ Acc: {acc:.4f}, Recall: {recall:.4f}, Score: {avg_score:.4f}")

    if avg_score > best_score:
        best_score = avg_score
        best_model = model

print("\n🎯 Best model score:", best_score)



🔍 Testing conv_layers=1, fc_units=128, lr=0.01, dropout=0.0, batch_size=32


✅ Acc: 0.7175, Recall: 0.0000, Score: 0.3588

🔍 Testing conv_layers=1, fc_units=128, lr=0.01, dropout=0.3, batch_size=32


✅ Acc: 0.7175, Recall: 0.0000, Score: 0.3588

🔍 Testing conv_layers=1, fc_units=128, lr=0.001, dropout=0.0, batch_size=32


✅ Acc: 0.8135, Recall: 0.7080, Score: 0.7607

🔍 Testing conv_layers=1, fc_units=128, lr=0.001, dropout=0.3, batch_size=32


✅ Acc: 0.8025, Recall: 0.5080, Score: 0.6552

🔍 Testing conv_layers=1, fc_units=256, lr=0.01, dropout=0.0, batch_size=32


...

✅ Acc: 0.8185, Recall: 0.5752, Score: 0.6969

🔍 Testing conv_layers=3, fc_units=256, lr=0.01, dropout=0.0, batch_size=32


✅ Acc: 0.7655, Recall: 0.2814, Score: 0.5235

🔍 Testing conv_layers=3, fc_units=256, lr=0.01, dropout=0.3, batch_size=32


✅ Acc: 0.7665, Recall: 0.3947, Score: 0.5806

🔍 Testing conv_layers=3, fc_units=256, lr=0.001, dropout=0.0, batch_size=32


✅ Acc: 0.7970, Recall: 0.6425, Score: 0.7197

🔍 Testing conv_layers=3, fc_units=256, lr=0.001, dropout=0.3, batch_size=32


✅ Acc: 0.8115, Recall: 0.5699, Score: 0.6907

🎯 Best model score: 0.7790464601769911
